In [ ]:
!gdal_translate -projwin 10 34 31 49 HYP_HR_SR_OB_DR.tif balkan.tif

In [ ]:
from matplotlib.patches import Polygon
from scipy.spatial import ConvexHull
import numpy as np

def add_group_outline(ax, df, group_col, region_colors, exclude_groups=None,
                      linestyle='--', linewidth=2, padding=0.05):
    if exclude_groups is None:
        exclude_groups = []

    for region, group in df.groupby(group_col):
        if region in exclude_groups or len(group) < 3:
            continue

        color = region_colors.get(region, "gray")
        coords = group[["Longitude", "Latitude"]].values

        try:
            # Get the convex hull of the coordinates
            hull = ConvexHull(coords)
            hull_pts = coords[hull.vertices]

            # Calculate the centroid of the hull points
            centroid = np.mean(hull_pts, axis=0)

            # Expand the hull points away from the centroid
            inflated_pts = []
            for pt in hull_pts:
                direction = pt - centroid
                direction /= np.linalg.norm(direction)  # Normalize to unit vector
                inflated_pts.append(pt + direction * padding)  # Inflate outward

            # Convert the inflated points to a numpy array
            inflated_pts = np.array(inflated_pts)

            # Create and add the polygon
            poly = Polygon(inflated_pts, closed=True, fill=False, edgecolor=color,
                           linewidth=linewidth, linestyle=linestyle,
                           transform=ccrs.PlateCarree(), zorder=3)
            ax.add_patch(poly)
        except Exception as e:
            print(f"Could not draw hull for {region}: {e}")


In [ ]:
import rasterio
from rasterio.windows import from_bounds
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from PIL import Image
Image.MAX_IMAGE_PIXELS = 570250000

# Define crop extent in WGS84 (lon/lat)
crop_extent = [13, 30, 34.5, 46.5]  # [min lon, max lon, min lat, max lat]
left, right = crop_extent[0], crop_extent[1]
bottom, top = min(crop_extent[2], crop_extent[3]), max(crop_extent[2], crop_extent[3])

# Path to your raster
fname = '/mnt/archgen/users/xiaowen/Kamenice/1024/map/HYP_HR_SR_OB_DR.tif'

# Open and crop raster
with rasterio.open(fname) as src:
    window = from_bounds(left, bottom, right, top, transform=src.transform)
    window = window.round_offsets().round_lengths()
    img = src.read(window=window)
    img_rgb = np.transpose(img, (1, 2, 0))
    if img_rgb.dtype != np.uint8:
        img_rgb = img_rgb.astype(np.float32)
        img_rgb /= img_rgb.max()

import pandas as pd

# Load first site CSV
site_df = pd.read_csv('Site_NM.csv')  # Replace with real path

# Site Id -> color
group_colors = {
    'KNC': u'#8c564b',
    'BCI': u'#bcbd22',
    'DDL': u'#ff7f0e',
    'GCE': u'#f72585',
    'MLI': u'#ffd60a',
    'PGN': u'#003049',
    'PLO': u'#1f77b4',
    'SND': u'#9e0059',
    'VRD': u'#17becf',
    'VRV': u'#219ebc',
    'MLQ': u'#ffb703'
}

# Load second CSV
extra_df = pd.read_csv('Reference_site_map2.csv')  # Replace with real path

# Define color mapping for second CSV
region_colors = {
    'CSB': u'#9467bd',
    'Dalmatia': u'#023e8a',
    'Turkey': u'#730220',
    'Greece': u'#003049',
    'Bulgaria': u'#e377c2',
    'NWB': u'#ff7f0e'
}

# Plot base map
fig = plt.figure(figsize=(12, 8), dpi=600)
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.set_extent(crop_extent, crs=ccrs.PlateCarree())

ax.imshow(img_rgb, origin='upper', transform=ccrs.PlateCarree(),
          extent=[left, right, bottom, top], interpolation='none')

# === Plot Site_NM.csv with jitter ===
lon_jitter = 0.15
lat_jitter = 0.15
used_coords = {}
jittered_sites = []

for site_id, group in site_df.groupby("Site Id"):
    color = group_colors.get(site_id, "black")
    zorder = 5  # Default zorder

    # Set zorder for KNC to be higher than MLQ to ensure KNC is on top
    if site_id == 'KNC':
        zorder = 6  # KNC should be on top

    for _, row in group.iterrows():
        lon = row["Longitude"]
        lat = row["Latitude"]
        key = (round(lon, 1), round(lat, 1))
        count = used_coords.get(key, 0)
        if count > 0:
            jittered_lon = lon + np.random.uniform(-lon_jitter, lon_jitter) * count
            jittered_lat = lat + np.random.uniform(-lat_jitter, lat_jitter) * count
            jittered_sites.append(f"{row['Name']} ({site_id})")
        else:
            jittered_lon, jittered_lat = lon, lat
        used_coords[key] = count + 1
        ax.scatter(jittered_lon, jittered_lat,
                   color=color, edgecolor='black', s=80, zorder=zorder, transform=ccrs.PlateCarree(),
                   label=f"{row['Name']} ({site_id})" if count == 0 else None)

# === Plot Other_Sites.csv ===
for _, row in extra_df.iterrows():
    lon = row["Longitude"]
    lat = row["Latitude"]
    region = row["color"]
    color = region_colors.get(region, "gray")
    ax.scatter(lon, lat,
               color=color, s=50, edgecolor='none', zorder=4, transform=ccrs.PlateCarree())

# Print jittered sites
if jittered_sites:
    print("Sites that were jittered due to overlapping coordinates:")
    for site in jittered_sites:
        print("-", site)
else:
    print("No overlapping sites detected. No jitter applied.")

# Create legend for Site_NM.csv
from matplotlib.lines import Line2D
legend_entries = {
    site_id: (group_colors.get(site_id, "black"), group["Name"].iloc[0])
    for site_id, group in site_df.groupby("Site Id")
}

handles = [
    Line2D([0], [0], marker='o', color='black', label=f"{name} ({site_id})",
           markerfacecolor=color, markersize=10, linewidth=0)
    for site_id, (color, name) in legend_entries.items()
]

# Assign "CSB" group to all rows in site_df
site_df["color"] = "CSB"

# Merge the two datasets
combined_df = pd.concat([extra_df, site_df], ignore_index=True)

#add_group_outline(ax, combined_df, group_col="color", region_colors=region_colors, exclude_groups=["Turkey"])


add_group_outline(ax, combined_df, group_col="color", region_colors=region_colors,
                  exclude_groups=[], padding=0.25)



# === Create a separate legend figure ===
from matplotlib.lines import Line2D

legend_entries = {
    site_id: (group_colors.get(site_id, "black"), group["Name"].iloc[0])
    for site_id, group in site_df.groupby("Site Id")
}

# Sort legend entries with KNC on top
sorted_entries = sorted(legend_entries.items(), key=lambda x: (x[0] != 'KNC', x[0]))

# Create handles
handles = [
    Line2D([0], [0], marker='o', color='black', label=f"{name} ({site_id})",
           markerfacecolor=color, markersize=10, linewidth=0)
    for site_id, (color, name) in sorted_entries
]

# Create a new figure for the legend
legend_fig = plt.figure(figsize=(4, len(handles) * 0.4),dpi=600)
legend_ax = legend_fig.add_subplot(111)
legend_ax.axis('off')  # Hide axes

# Add legend to this figure
legend = legend_ax.legend(handles=handles, loc='center left', frameon=True,
                          fontsize=9, title='Sites', title_fontsize=10)
legend_fig.savefig("legend.svg", format='svg', bbox_inches='tight')
# Save or show the legend figure
plt.tight_layout()
plt.show()

